# C2U in RAG — Pipeline Thực Nghiệm Kaggle (GĐ1 + GĐ2 + GĐ4)

**Cấu hình bắt buộc trước khi chạy:**
1. **Accelerator:** `GPU T4 x2` hoặc `GPU P100`
2. **Internet:** `On` (để tải weights encoder từ HuggingFace)
3. **Add Input:** Thêm dataset chứa `c2u_full_bundle.zip` (ví dụ `c2u-bundle`)

In [ ]:
# CELL 1: Kiểm tra GPU và cài đặt thư viện cần thiết
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        vram_gb = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
        print(f"GPU {idx}: {torch.cuda.get_device_name(idx)} ({vram_gb:.2f} GB VRAM)")

!pip install -q sentence-transformers concept-erasure rank_bm25 scikit-learn pyyaml tqdm

In [ ]:
# CELL 2: Nạp mã nguồn, splits sang /kaggle/working và phục hồi backup (nếu có)
import os
import shutil
import glob
import zipfile

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

WORK_DIR = "/kaggle/working"
items_to_copy = ["src", "configs", "manifests", "processed_splits", "tests", "run_kaggle.py", "requirements.txt"]

# Tìm thư mục đã giải nén sẵn hoặc file c2u_full_bundle.zip trong /kaggle/input
src_input_dir = None
bundle_zip_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "src" in dirs and "processed_splits" in dirs:
        src_input_dir = root
        break
    if "c2u_full_bundle.zip" in files:
        bundle_zip_path = os.path.join(root, "c2u_full_bundle.zip")

if src_input_dir:
    print(f"Tìm thấy mã nguồn tại: {src_input_dir}")
    for item in items_to_copy:
        src_path = os.path.join(src_input_dir, item)
        dst_path = os.path.join(WORK_DIR, item)
        if os.path.exists(src_path):
            if os.path.isdir(src_path):
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)
            print(f"  -> Đã copy {item}")
elif bundle_zip_path:
    print(f"Tìm thấy bundle nén tại: {bundle_zip_path}. Đang giải nén...")
    with zipfile.ZipFile(bundle_zip_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print("  -> Đã giải nén hoàn tất vào /kaggle/working")
else:
    raise FileNotFoundError("Không tìm thấy thư mục src/processed_splits hoặc file c2u_full_bundle.zip trong /kaggle/input!")

os.makedirs(os.path.join(WORK_DIR, "cache_embeddings"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "results"), exist_ok=True)

# Phục hồi backup checkpoints/cache nếu có file *results*.zip trong /kaggle/input hoặc /kaggle/working
backup_zips = [
    p for p in glob.glob("/kaggle/working/*results*.zip") + glob.glob("/kaggle/input/**/*results*.zip", recursive=True)
    if os.path.basename(p) != "c2u_full_bundle.zip"
]
if backup_zips:
    target_zip = backup_zips[0]
    print(f"\n-> Phát hiện file backup: {target_zip}. Đang giải nén...")
    with zipfile.ZipFile(target_zip, "r") as zf:
        zf.extractall(WORK_DIR)
    n_ckpts = len(glob.glob(os.path.join(WORK_DIR, "checkpoints", "*.pt")))
    n_caches = len(glob.glob(os.path.join(WORK_DIR, "cache_embeddings", "*.pt")))
    print(f"-> Đã phục hồi {n_ckpts} checkpoints và {n_caches} cache embeddings.")
else:
    print("\n-> Không có file backup cũ, sẽ chạy từ đầu.")

# LƯU Ý: Nếu backup cũ có chứa *_qwen3.pt được encode trước bản vá double-prompt,
# hãy bỏ dấu # ở dòng dưới để xóa riêng cache qwen3 cũ và encode lại cho chuẩn:
# !rm -f /kaggle/working/cache_embeddings/*_qwen3.pt

In [ ]:
# CELL 3: Kiểm tra toàn vẹn 15 splits & khả năng nạp toán tử C2U v2
%cd /kaggle/working
!python tests/test_data_integrity.py
!python -c "from src.c2u_operator import C2UOperator, C2UComposeOperator, AvgDirectionOperator; print('C2U v2 import OK')"

In [ ]:
# CELL 4: Chạy GĐ1 (Cache embeddings) + GĐ2 (Train ACR) cho đủ 60 units (48 đơn + 12 arxiv_multi)
%cd /kaggle/working
!python run_kaggle.py --dataset all --seed 0 --encoder all --max_epochs 10
# Nếu muốn lưu thêm mốc tái lập 1 epoch (_ep1.pt), thêm cờ --repro_epoch1 vào cuối lệnh trên.

In [ ]:
# CELL 5: Chạy toàn bộ đánh giá GĐ4 v2 (chỉ trên selection & confirm, khóa tập test)
%cd /kaggle/working
import subprocess
import yaml

# 5.1 Tái lập Dense C2U & Baselines, tự động chọn và khóa gate_quantile_space
!python src/evaluate_dense.py --datasets tofu amazon mnli arxiv --seeds 1 2 3 --encoders minilm bge e5 qwen3 --stage both --gate_quantile_space auto

with open("configs/dense_lock.yaml", "r", encoding="utf-8") as f:
    locked_space = yaml.safe_load(f).get("gate_quantile_space", "query")
print(f"\n[Locked gate_quantile_space] = {locked_space}")

# 5.2 Biến đổi truy vấn một lượt (12 ô = 4 dataset x 3 điều kiện)
subprocess.run([
    "python", "src/evaluate_transforms.py",
    "--datasets", "tofu", "amazon", "mnli", "arxiv",
    "--seeds", "1", "2", "3",
    "--encoders", "minilm", "bge", "e5", "qwen3",
    "--stage", "confirm",
    "--gate_quantile_space", locked_space,
], check=True)

# 5.3 Đa khái niệm ArXiv K=4 và K=2 đối chứng
subprocess.run([
    "python", "src/evaluate_multi.py",
    "--seeds", "1", "2", "3",
    "--encoders", "minilm", "bge", "e5", "qwen3",
    "--stage", "confirm",
    "--gate_quantile_space", locked_space,
], check=True)

# 5.4 Ablation vai trò ACR trên selection
subprocess.run([
    "python", "src/ablation_acr.py",
    "--datasets", "tofu", "amazon", "mnli", "arxiv",
    "--seeds", "1", "2", "3",
    "--encoders", "minilm", "bge", "e5", "qwen3",
    "--gate_quantile_space", locked_space,
], check=True)

In [ ]:
# CELL 6: Đóng gói toàn bộ kết quả GĐ4, checkpoints và cache để tải về máy
%cd /kaggle/working
import os
import zipfile

output_zip = "/kaggle/working/c2u_results_and_checkpoints.zip"
total_files = 0
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for target_dir in ["checkpoints", "cache_embeddings", "results", "configs", "manifests"]:
        if os.path.exists(target_dir):
            for root, dirs, files in os.walk(target_dir):
                for fn in files:
                    fp = os.path.join(root, fn)
                    zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
                    total_files += 1

size_mb = os.path.getsize(output_zip) / (1024 * 1024)
print(f"[Done] Đã đóng gói {total_files} files vào {output_zip} ({size_mb:.2f} MB)")